# Three levels of user-defined equations

An operator at one scale can be another `MultiscaleProblem`. This small
coefficient example declares every A, B, C and D block. It verifies recursive
field reconstruction against a separately written full matrix; it is an
algebraic example, not a convergence study for a physical PDE.

At the leaf, $u=(u_1,u_2)$ and the trace is $\xi$. Middle and top traces
are $\eta$ and $\lambda$. Their equations are

$$
\begin{aligned}
\begin{pmatrix}2&-1\\-1&2\end{pmatrix}u
  +\begin{pmatrix}1\\0\end{pmatrix}\xi &= \begin{pmatrix}1\\3\end{pmatrix},\\
-u_1+3\xi+2\eta &= 2,\\
-3\xi+4\eta+\lambda &= 1,\\
-2\eta+6\lambda &= 3.
\end{aligned}
$$

No symmetry relation between B and C is assumed. Intermediate operators are
the actually assembled finer systems; the final solution retains their tree.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble

def leaf_equations(item):
    """Supply the finest two coefficients and their single interface balance."""
    return LocalEquations([[2.0, -1.0], [-1.0, 2.0]], [1.0, 3.0],
                          [[1.0], [0.0]], [[-1.0, 0.0]], [0], d=[[3.0]], g=[2.0])

leaf = MultiscaleProblem(Equation(0, 0), leaf_equations, [0], 1, (0,))

def middle_equations(item):
    """Use the leaf's reduced operator as this level's local A block."""
    return LocalEquations(leaf, 0, [[2.0]], [[-3.0]], [0], d=[[4.0]], g=[1.0])

middle = MultiscaleProblem(Equation(0, 0), middle_equations, [0], 1, (0,))

def top_equations(item):
    """Use the middle system without flattening or recomputing its leaf basis."""
    return LocalEquations(middle, 0, [[1.0]], [[-2.0]], [0], d=[[6.0]], g=[3.0])

problem = MultiscaleProblem(Equation(0, 0), top_equations, [0], 1, (0,))
system = assemble(problem)
solution = system.solve()
child = solution.children[0]
finest = child.children[0]
recovered = np.r_[finest.fields[0], finest.trace, child.trace, solution.trace]

monolithic = np.array([[2., -1., 1., 0., 0.], [-1., 2., 0., 0., 0.],
                       [-1., 0., 3., 2., 0.], [0., 0., -3., 4., 1.],
                       [0., 0., 0., -2., 6.]])
rhs = np.array([1., 3., 2., 1., 3.])
reference = np.linalg.solve(monolithic, rhs)
np.testing.assert_allclose(recovered, reference, atol=2e-14, rtol=2e-14)
print({"reconstructed_coefficients": recovered.tolist(),
       "monolithic_equation_residual": float(np.linalg.norm(monolithic @ recovered - rhs)),
       "relative_residuals_by_level": [solution.raw_residual, child.raw_residual,
                                        finest.raw_residual]})
